# Facial-Emotion Recognition — Model Training
**Module:** Facial-Emotion Recognition (Member 2)

**Workflow:** Load dataset → detect faces / extract frames → preprocess → train CNN on 4 emotion categories (happy, sad, excited, neutral) → evaluate → convert to TensorFlow Lite for on-device deployment.

**Dataset:** Kaggle — `fahadullaha/facial-emotion-recognition-dataset`

Run cells top to bottom in Google Colab (Runtime → Change runtime type → GPU).

## 1. Setup and Kaggle authentication

In [ ]:
!pip install -q kaggle opencv-python-headless tensorflow scikit-learn matplotlib seaborn

import os, shutil, zipfile
from google.colab import files

print("Upload your kaggle.json (Kaggle account -> Create New API Token)")
uploaded = files.upload()  # upload kaggle.json here

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
!kaggle datasets download -d fahadullaha/facial-emotion-recognition-dataset -p /content/data --unzip

DATA_ROOT = '/content/data'
for root, dirs, fnames in os.walk(DATA_ROOT):
    depth = root.replace(DATA_ROOT, '').count(os.sep)
    if depth < 2:
        print(root, '->', dirs)

## 2. Inspect classes and map to the 4 target categories
The raw dataset may ship with the standard FER labels (angry, disgust, fear, happy, neutral, sad, surprise).
We map those down to our four project categories: **happy, sad, excited, neutral**.

> Check the printed folder names above and adjust `CLASS_MAP` below to match exactly what you see.

In [ ]:
# Adjust the keys on the left to match the ACTUAL folder/class names printed above
CLASS_MAP = {
    'happy':    'happy',
    'sad':      'sad',
    'surprise': 'excited',   # surprise -> excited
    'neutral':  'neutral',
    # classes not needed for this project (uncomment/edit as required):
    # 'angry':  None,
    # 'disgust':None,
    # 'fear':   None,
}

TARGET_CLASSES = ['happy', 'sad', 'excited', 'neutral']
print('Target classes:', TARGET_CLASSES)

## 3. Face detection + frame extraction (OpenCV)

In [ ]:
import cv2
import numpy as np

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
IMG_SIZE = 96

def detect_and_crop_face(img_path, size=IMG_SIZE):
    """Load an image, detect the largest face, crop, resize and grayscale-normalize it.
    Falls back to the full image (resized) if no face is detected — common for
    already-cropped FER datasets."""
    img = cv2.imread(img_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
    if len(faces) > 0:
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        face = gray[y:y+h, x:x+w]
    else:
        face = gray
    face = cv2.resize(face, (size, size))
    face = face.astype('float32') / 255.0
    return face

## 4. Build the dataset (images -> arrays)

In [ ]:
X, y = [], []
label_to_idx = {c: i for i, c in enumerate(TARGET_CLASSES)}

# Adjust this path to wherever the extracted dataset's class folders live
# e.g. /content/data/train/<class_name>/*.jpg
SEARCH_ROOT = DATA_ROOT

for src_class, target_class in CLASS_MAP.items():
    if target_class is None:
        continue
    for root, dirs, fnames in os.walk(SEARCH_ROOT):
        if os.path.basename(root).lower() == src_class.lower():
            for fn in fnames:
                if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
                    face = detect_and_crop_face(os.path.join(root, fn))
                    if face is not None:
                        X.append(face)
                        y.append(label_to_idx[target_class])

X = np.array(X).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = np.array(y)
print('Total samples:', X.shape[0])
print('Per-class counts:', {c: int((y == i).sum()) for c, i in label_to_idx.items()})

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

y_cat = to_categorical(y, num_classes=len(TARGET_CLASSES))
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=42, stratify=y)

print('Train:', X_train.shape, ' Test:', X_test.shape)

## 5. CNN model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 1), num_classes=4):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_cnn(num_classes=len(TARGET_CLASSES))
model.summary()

## 6. Train

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)
datagen.fit(X_train)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_test, y_test),
    epochs=30,
    callbacks=[early_stop]
)

## 7. Evaluate — accuracy/loss curves + confusion matrix

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].plot(history.history['accuracy'], label='train')
axs[0].plot(history.history['val_accuracy'], label='val')
axs[0].set_title('Accuracy'); axs[0].legend()

axs[1].plot(history.history['loss'], label='train')
axs[1].plot(history.history['val_loss'], label='val')
axs[1].set_title('Loss'); axs[1].legend()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred = model.predict(X_test).argmax(axis=1)
y_true = y_test.argmax(axis=1)

print(classification_report(y_true, y_pred, target_names=TARGET_CLASSES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.show()

## 8. Save model and convert to TensorFlow Lite

In [ ]:
model.save('/content/fer_model.h5')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('/content/fer_model.tflite', 'wb') as f:
    f.write(tflite_model)

print('Saved: fer_model.h5 and fer_model.tflite')
print('TFLite size (KB):', len(tflite_model) / 1024)

In [ ]:
from google.colab import files
files.download('/content/fer_model.tflite')  # download to add into the Android app's assets/ folder

## 9. Quick sanity check — run the TFLite model on one test image

In [ ]:
interpreter = tf.lite.Interpreter(model_path='/content/fer_model.tflite')
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample = X_test[0:1].astype('float32')
interpreter.set_tensor(input_details[0]['index'], sample)
interpreter.invoke()
pred = interpreter.get_tensor(output_details[0]['index'])

print('Predicted class:', TARGET_CLASSES[pred.argmax()])
print('Actual class:', TARGET_CLASSES[y_test[0].argmax()])
plt.imshow(X_test[0].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
plt.title(f"Pred: {TARGET_CLASSES[pred.argmax()]}")
plt.axis('off')
plt.show()

---
### What to show your guide from this notebook
1. The dataset download cell — proves the real Kaggle dataset was used.
2. The per-class sample counts (Step 4) — real numbers, not made up.
3. The accuracy/loss curves and confusion matrix (Step 7) — genuine training result.
4. The saved `fer_model.tflite` file and its size — the actual deployable model.
5. Step 9's live prediction on a test image — shows the model actually works, end to end.

### Next step for Android integration
Copy `fer_model.tflite` into `app/src/main/assets/`, add the `org.tensorflow:tensorflow-lite` dependency to `build.gradle`, and load it with `Interpreter` inside a new `EmotionDetector.java` / `EmotionDetectionFragment.java` — matching what the report already describes as "embedded into the Android application project."